In [35]:
from qiskit.converters import circuit_to_dag, dag_to_circuit
from qiskit.transpiler.passes import *
from qiskit import QuantumCircuit
from qiskit.circuit import Qubit
from qiskit.dagcircuit import DAGCircuit
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
    OPT_PASSES = [
        CommutativeCancellation,
        CommutativeInverseCancellation,
        # ElidePermutations,
        InverseCancellation,
        Optimize1qGates,
        Optimize1qGatesSimpleCommutation,
        OptimizeSwapBeforeMeasure,
        # RemoveDiagonalGatesBeforeMeasure,
        # RemoveFinalReset,
        RemoveIdentityEquivalent,
    ]

In [55]:
qc = QuantumCircuit.from_qasm_file('./data/20Q_gate_Tokyo/circuits/20Q_gate_Tokyo_large_1_10_1.5_no.0.qasm')
dag = circuit_to_dag(qc)

In [56]:
dag.count_ops()

{'cx': 207, 'h': 312}

In [57]:
sum(dag.count_ops().values()), dag.size()

(519, 519)

In [58]:
opt = CommutativeCancellation()
new_dag = opt.run(dag)

In [59]:
sum(dag.count_ops().values()), dag.size()

(277, 277)

In [60]:
sum(new_dag.count_ops().values()), new_dag.size()

(277, 277)

这就是为什么对resulting dag做Transform，前后的操作数量总是一样，相关Reward=0，但是Agent还是会继续采取这个动作，
因为也是用了Reward shaping，把之前dag的ops记录下来了，这样还是能得到奖励。

In [62]:
True | False

True

In [48]:
r = dag.count_ops()
r['total'] = sum(r.values())
r['trans'] = 'orig'
records = [r]

for opt_class in OPT_PASSES:
    opt = opt_class()
    new_dag = opt.run(dag)
    print(opt_class, new_dag.count_ops())
    r = new_dag.count_ops()
    r['total'] = sum(r.values())
    r['trans'] = opt_class.__name__
    records.append(r)

df = pd.DataFrame.from_records(records)
df

<class 'qiskit.transpiler.passes.optimization.commutative_cancellation.CommutativeCancellation'> {'cx': 201, 'h': 76}
<class 'qiskit.transpiler.passes.optimization.commutative_inverse_cancellation.CommutativeInverseCancellation'> {'cx': 193, 'h': 74}
<class 'qiskit.transpiler.passes.optimization.inverse_cancellation.InverseCancellation'> {'cx': 193, 'h': 74}
<class 'qiskit.transpiler.passes.optimization.optimize_1q_gates.Optimize1qGates'> {'cx': 193, 'h': 74}
<class 'qiskit.transpiler.passes.optimization.optimize_1q_commutation.Optimize1qGatesSimpleCommutation'> {'cx': 193, 'h': 74}
<class 'qiskit.transpiler.passes.optimization.optimize_swap_before_measure.OptimizeSwapBeforeMeasure'> {'cx': 193, 'h': 74}
<class 'qiskit.transpiler.passes.optimization.remove_identity_equiv.RemoveIdentityEquivalent'> {'cx': 193, 'h': 74}


,cx,h,total,trans
0,207,312,519,orig
1,201,76,277,CommutativeCancellation
2,193,74,267,CommutativeInverseCancellation
3,193,74,267,InverseCancellation
4,193,74,267,Optimize1qGates
5,193,74,267,Optimize1qGatesSimpleCommutation
6,193,74,267,OptimizeSwapBeforeMeasure
7,193,74,267,RemoveIdentityEquivalent


In [50]:
new_dag.size()

267

In [22]:
trans_list = [cls() for cls in OPT_PASSES]

In [23]:
{(0, trans): i for i, trans in enumerate(trans_list)}

{(0,
  <qiskit.transpiler.passes.optimization.commutative_cancellation.CommutativeCancellation at 0x150208440>): 0,
 (0,
  <qiskit.transpiler.passes.optimization.commutative_inverse_cancellation.CommutativeInverseCancellation at 0x15020acc0>): 1,
 (0,
  <qiskit.transpiler.passes.optimization.inverse_cancellation.InverseCancellation at 0x150209ee0>): 2,
 (0,
  <qiskit.transpiler.passes.optimization.optimize_1q_gates.Optimize1qGates at 0x150208140>): 3,
 (0,
  <qiskit.transpiler.passes.optimization.optimize_1q_commutation.Optimize1qGatesSimpleCommutation at 0x1502088c0>): 4,
 (0,
  <qiskit.transpiler.passes.optimization.optimize_swap_before_measure.OptimizeSwapBeforeMeasure at 0x150209580>): 5,
 (0,
  <qiskit.transpiler.passes.optimization.remove_identity_equiv.RemoveIdentityEquivalent at 0x150208500>): 6}

In [51]:
for op_node in dag.op_nodes():
    print(dag.size(), new_dag.size())
    dag.remove_op_node(op_node)


267 267
266 266
265 265
264 264
263 263
262 262
261 261
260 260
259 259
258 258
257 257
256 256
255 255
254 254
253 253
252 252
251 251
250 250
249 249
248 248
247 247
246 246
245 245
244 244
243 243
242 242
241 241
240 240
239 239
238 238
237 237
236 236
235 235
234 234
233 233
232 232
231 231
230 230
229 229
228 228
227 227
226 226
225 225
224 224
223 223
222 222
221 221
220 220
219 219
218 218
217 217
216 216
215 215
214 214
213 213
212 212
211 211
210 210
209 209
208 208
207 207
206 206
205 205
204 204
203 203
202 202
201 201
200 200
199 199
198 198
197 197
196 196
195 195
194 194
193 193
192 192
191 191
190 190
189 189
188 188
187 187
186 186
185 185
184 184
183 183
182 182
181 181
180 180
179 179
178 178
177 177
176 176
175 175
174 174
173 173
172 172
171 171
170 170
169 169
168 168
167 167
166 166
165 165
164 164
163 163
162 162
161 161
160 160
159 159
158 158
157 157
156 156
155 155
154 154
153 153
152 152
151 151
150 150
149 149
148 148
147 147
146 146
145 145
144 144
143 143


In [52]:
dag.size()

0

In [53]:
new_dag.size()

0

In [42]:
new_dag.count_ops()

{}